In [1]:
import os
from autogen.agentchat import UserProxyAgent, AssistantAgent, GroupChat, GroupChatManager
from autogen.coding import LocalCommandLineCodeExecutor
from dotenv import load_dotenv
from openai import AzureOpenAI
import json
import pandas as pd
import numpy as np
from datetime import datetime
load_dotenv()

azure_gpt4o = {
    "api_type": "azure",
    "model": os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'),
    "api_key": os.getenv('OPENAI_API_KEY'),
    "base_url": os.getenv('AZURE_OPENAI_ENDPOINT'),
    "api_version": os.getenv('OPENAI_API_VERSION')
}

flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.


In [2]:
#print(os.getenv('OPENAI_API_VERSION'))
blandai_data_dir = os.path.join(os.getcwd(), 'blandai-data')
codebook_file = os.path.join(os.getcwd(), 'transcripts/codebook.json')
transcripts_file = os.path.join(blandai_data_dir, 'blandai_transcripts_2025-01-06.json')
#sample_output_file = os.path.join(data_dir, 'example_output.csv')
#print(sample_output_file)
#print(codebook_file)
#print(blandai_data_dir)
#print(transcripts_file)

In [35]:
def get_QA_and_codebook(codebook_file):
    qa_list = []
    with open(codebook_file, 'r') as file:
        codebook = json.load(file)
        question_count = 1
        for code, content in codebook.items():  
            question = content['question']
            qa_list.append(f"Question {question_count}: {question}\nResponse Options: ")
            ans_list = []
            for ans, ans_id in content['clean_response_text_to_id'].items():
                ans_list.append(f"{ans}")
            #if code in ['HH01S', 'HH25S', 'HH612S', 'HH1317S', 'HH18OVS', 'PHYS11_TEMP']:
            #    ans_list = ['Numeric Value']
                
            qa_list.append('; '.join(ans_list))
            qa_list.append("\n")
            question_count += 1
    return ''.join(qa_list), codebook
QA_details, codebook = get_QA_and_codebook(codebook_file)
print(QA_details)

Question 1: What is your current age?
Response Options: 18-24; 25-34; 35-44; 45-54; 55-64; 65-74; 75+; Under 18
Question 2: Are you male or female
Response Options: Unknown; Male; Female; Not sure; REFUSED
Question 3: What race or races you consider yourself to be? You can say multiple races.
Response Options: White; Black or African American; American Indian or Alaska Native; Asian Indian; Chinese; Filipino; Japanese; Korean; Vietnamese; Other Asian; Native Hawaiian; Guamanian or Chamorro; Samoan; Other Pacific Islander; Some other race; REFUSED
Question 4: Household income
Response Options: Under $10,000; $10,000 to under $20,000; $20,000 to under $30,000; $30,000 to under $40,000; $40,000 to under $50,000; $50,000 to under $75,000; $75,000 to under $100,000; $100,000 to under $150,000; $150,000 or more; DON'T KNOW; REFUSED
Question 5: What is the highest level of school you have completed?
Response Options: No HS diploma; HIGH SCHOOL GRADUATE - high school DIPLOMA or the equiva; Som

In [37]:
client = AzureOpenAI(
  api_key = os.getenv('OPENAI_API_KEY'),  
  api_version = os.getenv('OPENAI_API_VERSION'),
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
)



role_description = f''' You are a helpful assistant that reads conversation transcript and deduces responses given by user to each question.
                        You return an output with the responses for each question in order of they appear in the question list.
                        The returned output needs to be a SINGLE LINE, with individual responses separated by semicolons and no other punctuation.
                        Format of the output line should be something like: Question 1: Response 1; Question 2: Response 2:; etc. 
                        Each deduced response for a question should be strictly selected from corresponding Reponse Options. 
                        If Reponse Options includes 'Numeric Value' as an option, deduce actual numeric value from conversation.
                        Each question should have an answer, if question has Numeric Value as an option you couldn't deduce the response put 'NaN'.
                        Make sure that number of answers equals number of questions.
                        
                        Here is the list of questions with respective comma separated response options:
                        {QA_details}
                        '''
#print(role_description)



with open(transcripts_file, 'r') as file:
    transcripts = json.load(file)

task_prompt = f"""Help me to understand the following conversation transcript : 

{transcripts["0"]}"""
#print(task_prompt)

conversation=[{"role": "system", "content": role_description}]
userid_to_answers = {}

for user_id, transcript in transcripts.items():
    task_prompt = f"""Help me to understand the following conversation transcript : 
                    {transcripts[user_id].replace('user:', 'surveyee:')}"""
    conversation.append({"role": "user", "content": task_prompt})

    #print(task_prompt)
    response = client.chat.completions.create(
        model=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'), # model = "deployment_name".
        messages=conversation
    )
    userid_to_answers[user_id] = response.choices[0].message.content
    print(userid_to_answers[user_id])
    conversation.pop()

Question 1: 35-44; Question 2: Female; Question 3: White; Question 4: $20,000 to under $30,000; Question 5: Some college, no degree; Question 6: Four persons; Question 7: 0; Question 8: 1; Question 9: 0; Question 10: 2; Question 11: 1; Question 12: Once a month; Question 13: Once a month; Question 14: Not at all or less than 1 day; Question 15: Not at all or less than 1 day; Question 16: Not at all or less than 1 day; Question 17: 1-2 days; Question 18: 1-2 days; Question 19: No, I did not work for pay last week; Question 20: Good; Question 21: Yes; Question 22: No; Question 23: Yes; Question 24: No; Question 25: Yes; Question 26: Yes; Question 27: No; Question 28: No; Question 29: Yes; Question 30: Yes; Question 31: Yes; Question 32: Yes; Question 33: 98.6
Question 1: 65-74; Question 2: Male; Question 3: White; Question 4: $150,000 or more; Question 5: Bachelors degree; Question 6: Two persons; Question 7: 0; Question 8: 0; Question 9: 0; Question 10: 0; Question 11: 2; Question 12: A

In [38]:
answers_as_list = []
for user_id in sorted(list(userid_to_answers.keys())):
    user_responses = [part.strip() for part in userid_to_answers[user_id].split(';')]
    clean_user_responses = []
    for response in user_responses:
        clean_text = response[response.index(':')+1:].strip()
        if clean_text == 'NaN':
            clean_user_responses.append('nan')
        else:
            clean_user_responses.append(clean_text)
    answers_as_list.append(clean_user_responses)
    print(len(clean_user_responses))
    print(clean_user_responses)

33
['35-44', 'Female', 'White', '$20,000 to under $30,000', 'Some college, no degree', 'Four persons', '0', '1', '0', '2', '1', 'Once a month', 'Once a month', 'Not at all or less than 1 day', 'Not at all or less than 1 day', 'Not at all or less than 1 day', '1-2 days', '1-2 days', 'No, I did not work for pay last week', 'Good', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'No', 'No', 'Yes', 'Yes', 'Yes', 'Yes', '98.6']
33
['65-74', 'Male', 'White', '$150,000 or more', 'Bachelors degree', 'Two persons', '0', '0', '0', '0', '2', 'A few times a month', 'A few times a month', 'Not at all or less than 1 day', 'Not at all or less than 1 day', 'Not at all or less than 1 day', 'Not at all or less than 1 day', 'Not at all or less than 1 day', 'No, I did not work for pay last week.', 'Very good', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', '98.5']
33
['65-74', 'Male', 'Some other race', '$150,000 or more', 'Bachelors degree', 'Two persons', '0', '0', '0', '0', '2', 'A fe

In [39]:
question_code_to_text_responses = {}
i = 0
#print(codebook['SOC1']['answer_to_answer_id']['Some'])
for code, val in codebook.items():
    print(code)
    question_code_to_text_responses[code] = []
    for user_answers in answers_as_list:
        response_text = user_answers[i]
        question_code_to_text_responses[code].append(response_text)
    i += 1

df = pd.DataFrame(question_code_to_text_responses)
df.head()

AGE7
GENDER
RACETH
HHINCOME
EDUCATION
HHSIZE1
HH01S
HH25S
HH612S
HH1317S
HH18OVS
SOC2A
SOC2B
SOC5A
SOC5B
SOC5C
SOC5D
SOC5E
ECON1
PHYS8
PHYS4
PHYS5
PHYS1B
PHYS1C
PHYS1D
PHYS1E
PHYS1F
PHYS1G
PHYS1H
PHYS1I
PHYS1J
PHYS11
PHYS11_TEMP


,AGE7,GENDER,RACETH,HHINCOME,EDUCATION,HHSIZE1,HH01S,HH25S,HH612S,HH1317S,...,PHYS1C,PHYS1D,PHYS1E,PHYS1F,PHYS1G,PHYS1H,PHYS1I,PHYS1J,PHYS11,PHYS11_TEMP
0,35-44,Female,White,"$20,000 to under $30,000","Some college, no degree",Four persons,0,1,0,2,...,No,Yes,Yes,No,No,Yes,Yes,Yes,Yes,98.6
1,65-74,Male,White,"$150,000 or more",Bachelors degree,Two persons,0,0,0,0,...,No,No,No,No,No,No,No,No,Yes,98.5
2,65-74,Male,Some other race,"$150,000 or more",Bachelors degree,Two persons,0,0,0,0,...,No,No,No,No,No,No,No,Yes,No,nan
3,65-74,Male,Black or African American,"$75,000 to under $100,000",Bachelors degree,Two persons,0,0,0,0,...,No,No,No,No,Yes,No,Yes,Yes,No,nan


In [40]:
gpt_res_dir = os.path.join(os.getcwd(),  'gpt-deductions')
if not os.path.exists(gpt_res_dir):
    os.makedirs(gpt_res_dir)

In [41]:
# Save conversation transcripts
fname  = os.path.join(gpt_res_dir,  f"deduced_{datetime.today().strftime('%Y-%m-%d')}.csv")
df.to_csv(fname)

In [42]:
idxs =[i for i in range(4)]
correct_results_df = pd.read_csv("./phone-survey-data/clean_text_full_phone_survey_01_April_30_covid_impact_survey.csv")
correct_results_df = correct_results_df.iloc[idxs]
correct_results_df.head()
print(correct_results_df['PHYS11_TEMP'].astype(str))
print(df['PHYS11_TEMP'].astype(str))
print()
print(df['PHYS11_TEMP'].astype(str) == correct_results_df['PHYS11_TEMP'].astype(str))

0    98.6
1    98.5
2     nan
3     nan
Name: PHYS11_TEMP, dtype: object
0    98.6
1    98.5
2     nan
3     nan
Name: PHYS11_TEMP, dtype: object

0    True
1    True
2    True
3    True
Name: PHYS11_TEMP, dtype: bool


In [43]:
for code in df.keys():
    predicted_series = df[code].astype(str)
    actual_series    = correct_results_df[code].astype(str)
    match = predicted_series == actual_series
    accuracy = match.sum()/len(match)
    print(code, accuracy)

AGE7 1.0
GENDER 1.0
RACETH 0.0
HHINCOME 1.0
EDUCATION 1.0
HHSIZE1 1.0
HH01S 1.0
HH25S 1.0
HH612S 1.0
HH1317S 1.0
HH18OVS 1.0
SOC2A 1.0
SOC2B 1.0
SOC5A 1.0
SOC5B 1.0
SOC5C 1.0
SOC5D 1.0
SOC5E 1.0
ECON1 0.5
PHYS8 1.0
PHYS4 1.0
PHYS5 1.0
PHYS1B 1.0
PHYS1C 1.0
PHYS1D 1.0
PHYS1E 1.0
PHYS1F 1.0
PHYS1G 1.0
PHYS1H 1.0
PHYS1I 1.0
PHYS1J 1.0
PHYS11 1.0
PHYS11_TEMP 1.0


In [9]:
question_code_to_answers = {}
i = 0
#print(codebook['SOC1']['answer_to_answer_id']['Some'])
for code, val in codebook.items():
    print(code)
    question_code_to_answers[code] = []
    for user_answers in answers_as_list:
        ans_text = user_answers[i]
        question_code_to_answers[code].append(codebook[code]['clean_response_text_to_id'][ans_text])
    i += 1

AGE7
GENDER
RACETH
HHINCOME
EDUCATION
HHSIZE1
HH01S


KeyError: '0'

In [ ]:
df = pd.DataFrame(question_code_to_answers)

In [ ]:
df.to_csv('deduced.csv')